In [26]:
import pandas as pd
import nltk
import re
import string


pd.set_option("display.max_columns", None)

In [27]:
df1 = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|")
df1.drop(["modern_prompt", "translation_prompt"], axis = 1, inplace=True)

df2 = pd.read_parquet("../../data/paires_mot_lemme.parquet")

/tmp/ipykernel_35343/1343399669.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df1 = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|")


In [28]:
df1.head(5)

,modern,old_french
0,"Salut tout le monde, c'est Victor. Je repensai...","Seigneurs et dames, je vous salue, c'est Victo..."
1,"Salut tout le monde ! Aujourd'hui, c'était un ...","Saluz tout le monde ! En ce jour, fu moult bon..."
2,"Chère Sophie, Tu ne devineras jamais ce qui m...","Chère Sophie, Tu ne devineras ja mie ce qui m..."
3,"Bonjour à tous, ici Marcel Dupré, artisan poti...","Bien le bon jour à tous, céans Marcel Dupré, o..."
4,"Yo, c’est Lila. Alors voilà, l’autre jour, j’é...","Salut, c'est Lila. Lors, l'autre jour, j'estoi..."


In [29]:
df2.head(5)

,mot,lemme
0,Cil,cil
1,qui,qui1
2,fist,faire
3,d',de
4,Erec,Erec


In [30]:
# fonctions

# lowercase
def text_lowercase(text):
    return text.lower()

# normalisation des espaces
def space_normalisation(text) :
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# suppression des emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)


# suppression des tags html
def remove_html_tag_urls(text) :
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'</?[a-zA-Z][a-zA-Z0-9]*\b[^>]*?>', '', text)
    return text

# supprimer mentions
def remove_mention(text) : 
    text = re.sub(r'@\w+', '', text)
    return text

# supprimer la ponctuation
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)


In [31]:
def pipeline(text) : 
    
    if pd.isna(text):
        return ""
    
    # suppression de la casse
    text = text_lowercase(text)
    
    # normalisation des espaces
    text = space_normalisation(text)
    
    # suppression emojis
    text = remove_emoji(text)
    
    # suppression des tags html
    text = remove_html_tag_urls(text)

    # supprimer mentions
    text = remove_mention(text)
    
    # # supprimer la ponctuation
    # text = remove_punctuation(text)
    
    return text


In [32]:
df1["modern"] = df1["modern"].apply(pipeline)
df1["old_french"] = df1["old_french"].apply(pipeline)

In [33]:
df1.head(5)

,modern,old_french
0,"salut tout le monde, c'est victor. je repensai...","seigneurs et dames, je vous salue, c'est victo..."
1,"salut tout le monde ! aujourd'hui, c'était un ...","saluz tout le monde ! en ce jour, fu moult bon..."
2,"chère sophie, tu ne devineras jamais ce qui m'...","chère sophie, tu ne devineras ja mie ce qui m'..."
3,"bonjour à tous, ici marcel dupré, artisan poti...","bien le bon jour à tous, céans marcel dupré, o..."
4,"yo, c’est lila. alors voilà, l’autre jour, j’é...","salut, c'est lila. lors, l'autre jour, j'estoi..."


In [37]:
df1.to_csv("../../data/cleaned_dataset.csv", sep="|", index=False)